# DriftEnv Colab Training Notebook

This notebook clones your GitHub repo, installs dependencies, starts the DriftEnv API server, and launches GRPO training.

In [ ]:
# ---- Configure these before running ----
GITHUB_REPO = "https://github.com/RaghavPrasanna9207/driftenv.git"
REPO_DIR = "driftenv"
HF_HUB_MODEL_ID = "RaghavPrasanna9207/driftenv-grpo"
HF_TOKEN = ""  # Paste your HF token (or set via Colab Secrets)
ENV_BASE_URL = "http://127.0.0.1:8000"
OUTPUT_DIR = "outputs/driftenv-grpo"

In [ ]:
!git clone {GITHUB_REPO}
%cd {REPO_DIR}
!python -m pip install --upgrade pip
!pip install -r requirements.txt

In [ ]:
# Optional: set token from variable if provided.
import os
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGINGFACE_HUB_TOKEN"] = HF_TOKEN
os.environ["HF_HUB_MODEL_ID"] = HF_HUB_MODEL_ID
os.environ["DRIFTENV_BASE_URL"] = ENV_BASE_URL

In [ ]:
# Start the DriftEnv API server in the notebook runtime.
import subprocess, time
server_proc = subprocess.Popen(
    ["python", "-m", "uvicorn", "server:app", "--host", "0.0.0.0", "--port", "8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
time.sleep(3)
print("Server PID:", server_proc.pid)

In [ ]:
# Quick health check
!curl -s http://127.0.0.1:8000/health

In [ ]:
# Run GRPO training (expects GPU runtime in Colab)
!python training/train_grpo.py \
  --env-base-url {ENV_BASE_URL} \
  --output-dir {OUTPUT_DIR} \
  --hub-model-id {HF_HUB_MODEL_ID} \
  --hf-token {HF_TOKEN}

In [ ]:
# Inspect training artifacts
!ls -R {OUTPUT_DIR}

In [ ]:
# Optional: stop server when done
server_proc.terminate()
server_proc.wait(timeout=10)
print("Server stopped")